# RAG Is Simpler Than You Think

- Six RAG recipes on the same 14 photos, simplest first
- One Pixeltable database holds the photos, captions, embeddings, questions, and answers
- The questions are text. The evidence is images.
- Add the next recipe only when the one before it misses

Recipe names and the idea come from Rafael Pierre's [RAG Is Simpler Than You Think](https://www.lighthousenewsletter.com/p/rag-is-simpler-than-you-think).

## Setup

- Run this notebook from `simpler-rag/` with the `simpler-rag` kernel
- Recipe 2 and the bonus call OpenAI: set `OPENAI_API_KEY` where the `pxt` daemon runs
- The cell after `uv sync` drops `rag_demo`, so every run starts from empty tables

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt

# Start fresh: drop the tables from any earlier run.
pxt.drop_table('rag_demo.queries', if_not_exists='ignore')  # first: it searches photos
pxt.drop_dir('rag_demo', force=True, if_not_exists='ignore')

## Recipe 1: The MVP – Full-Text Search Only

- A `photos` table: ID, image, title
- Keyword search on titles. No models, no API calls, no chunking.

In [ ]:
!cat recipe1_fulltext.py

In [ ]:
!pxt schema diff recipe1_fulltext.py rag_demo

In [ ]:
!pxt schema update recipe1_fulltext.py rag_demo -f

In [ ]:
BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images/'
PHOTOS = [  # (photo_id, image number, title)
    (1, 1, 'Taxi on a city street'),
    (2, 9, 'Bento lunch box'),
    (3, 16, 'Baseball batter'),
    (4, 25, 'Giraffe at the zoo'),
    (5, 30, 'Vase of flowers'),
    (6, 34, 'Zebra grazing'),
    (7, 42, 'Dog asleep on the shoes'),
    (8, 49, 'Horse riders performing'),
    (9, 57, 'Tennis match'),
    (10, 61, 'Elephant trek in the jungle'),
    (11, 64, 'Street clock'),
    (12, 71, 'Commuter train in the yard'),
    (13, 80, 'Zebra in an enclosure'),
    (14, 90, 'Cow in a pasture'),
]

photos = pxt.get_table('rag_demo.photos')
photos.insert(
    [{'photo_id': i, 'image': f'{BASE}{n:012d}.jpg', 'title': t} for i, n, t in PHOTOS]
)

In [ ]:
photos.where(photos.title.contains('zebra', case=False)).select(
    photos.photo_id, photos.image, photos.title
).collect()

In [ ]:
photos.where(photos.title.contains('puppy', case=False)).select(
    photos.photo_id, photos.image, photos.title
).collect()

- ✅ `zebra` finds both zebra photos
- ❌ `puppy` finds nothing. The photo is there, but its title calls it a dog.

## Recipe 2: Agentic Query Rewriting

- A new `queries` table stores each question someone asks, one row per question
- An LLM turns the question into keywords, and `first_hits` searches the titles
- The model then sees what it found. If nothing matched, it tries broader keywords.
- `title_hits` runs a search for each new question and saves the matching photos next to it
- Still no embeddings. If results miss, change the prompt.

In [ ]:
!cat recipe2_agentic.py

In [ ]:
!pxt schema diff recipe2_agentic.py rag_demo

In [ ]:
!pxt schema update recipe2_agentic.py rag_demo -f

### Ask questions

- People type questions, not keywords: `any puppies?`, not `dog`
- Each question is a row in `queries`. Inserting it runs both tries and both searches.

In [ ]:
queries = pxt.get_table('rag_demo.queries')
queries.insert(
    [
        {'raw': 'any puppies?'},
        {'raw': 'do you have a photo of the subway?'},
    ]
)
queries.select(
    queries.raw, queries.first_keywords, queries.first_hits, queries.keywords, queries.title_hits
).collect()

- ✅ When the first keywords miss, the second try adds words like `dog` and `train`
- ❌ Titles are short. Most of what a photo shows isn't in its title.

## Recipe 3: Hybrid Search (Sparse + Dense Reranking)

- Sparse: a BLIP caption for every photo, so keyword search has more words to match
- Dense: a CLIP embedding index on the images. CLIP puts images and text in one space.
- Both are declared in the schema. Pixeltable fills them for all 14 photos now, and for new photos on insert.
- The models download on the first run

In [ ]:
!cat recipe3_hybrid.py

In [ ]:
!pxt schema diff recipe3_hybrid.py rag_demo

In [ ]:
!pxt schema update recipe3_hybrid.py rag_demo -f

In [ ]:
photos = pxt.get_table('rag_demo.photos')  # reload to see the new column and index
photos.select(photos.photo_id, photos.image, photos.title, photos.caption).collect()

In [ ]:
sim = photos.image.similarity(string='a striped animal', idx='clip_idx')
photos.order_by(sim, asc=False).limit(3).select(
    photos.photo_id, photos.image, photos.title, photos.caption, score=sim
).collect()

### Keywords first, then rerank by meaning

In [ ]:
sim = photos.image.similarity(string='a jungle', idx='clip_idx')
photos.where(photos.caption.contains('riding')).order_by(sim, asc=False).select(
    photos.photo_id, photos.image, photos.title, photos.caption, score=sim
).collect()

- ✅ `a striped animal` ranks both zebras first. No title or caption says `striped`.
- ✅ `riding` finds the candidates. CLIP puts the elephant trek in the jungle above the horse riders.

## Recipe 4: On-The-Fly Embedding (The Fresh Data Play)

- The case for on-the-fly: stored embeddings go stale, and switching models means re-embedding everything
- In Pixeltable, though, his reasons for avoiding an index mostly disappear:
  - The index updates on every insert and update, so it is never stale
  - Only new or changed rows get embedded
  - A new model is a second index next to the first (Recipe 6)
- No schema change here. Insert two new photos and search right away.

In [ ]:
photos.insert(
    [
        {'photo_id': 15, 'image': f'{BASE}{36:012d}.jpg'},
        {'photo_id': 16, 'image': f'{BASE}{63:012d}.jpg'},
    ]
)
sim = photos.image.similarity(string='an umbrella', idx='clip_idx')
photos.order_by(sim, asc=False).limit(3).select(
    photos.photo_id, photos.image, photos.title, photos.caption, score=sim
).collect()

- ✅ Photo 15 has no title, and it's already captioned, embedded, and the top match for `an umbrella`

## Recipe 5: Pre-Embedding with Hot/Cold Tiers

- A `hot` column marks the photos people look at most
- The hot tier is a view: `photos` where `hot` is true
- Only the hot tier gets the newer CLIP model, so a model upgrade embeds a few photos, not all of them
- Moving a photo between tiers is an `update`. The view and its index follow.

In [ ]:
!cat recipe5_tiers.py

In [ ]:
!pxt schema diff recipe5_tiers.py rag_demo

In [ ]:
!pxt schema update recipe5_tiers.py rag_demo -f

In [ ]:
photos = pxt.get_table('rag_demo.photos')  # reload to see the new column
photos.update({'hot': True}, where=photos.photo_id.isin([4, 6, 7, 9]))

hot = pxt.get_table('rag_demo.hot_photos')
sim = hot.image.similarity(string='a striped animal', idx='hot_idx')
hot.order_by(sim, asc=False).select(hot.photo_id, hot.image, hot.title, score=sim).collect()

### Promote a photo

In [ ]:
photos.update({'hot': True}, where=photos.photo_id == 13)
hot.order_by(sim, asc=False).select(hot.photo_id, hot.image, hot.title, score=sim).collect()

- ✅ The hot tier holds 5 photos, and only those have the newer model's embedding
- ✅ Promoting photo 13 was one `update`. It's in the hot index right away.

## Recipe 6: Full Pre-Embedding (The Scale Play)

- Every photo gets the newer model, next to the old one. The old index keeps serving while the new one fills.
- Compare them on the same question
- Keep the winner and drop the old index

In [ ]:
!cat recipe6_compare.py

In [ ]:
!pxt schema diff recipe6_compare.py rag_demo

In [ ]:
!pxt schema update recipe6_compare.py rag_demo -f

In [ ]:
photos = pxt.get_table('rag_demo.photos')  # reload to see the new index

def top3(query, idx):
    sim = photos.image.similarity(string=query, idx=idx)
    return photos.order_by(sim, asc=False).limit(3).select(
        photos.photo_id, photos.image, photos.title, photos.caption, score=sim
    ).collect()

print('clip_idx (ViT-B/32)')
display(top3('a pet', 'clip_idx'))
print('clip_idx_v2 (ViT-B/16)')
display(top3('a pet', 'clip_idx_v2'))

- ✅ ViT-B/16 ranks the dog first
- ❌ ViT-B/32 ranks the two zebras above the dog

### Keep the winner

- `recipe6_cutover.py` keeps only `clip_idx_v2`
- The diff marks the drop `DESTRUCTIVE`. The update needs `--allow-destructive`.

In [ ]:
!pxt schema diff recipe6_cutover.py rag_demo

In [ ]:
!pxt schema update recipe6_cutover.py rag_demo --allow-destructive -f

- ✅ Model deprecation handled: add the new index, compare, drop the old one. No reloading the photos.

## Bonus: Multi-intent questions, answered from the photos

- Some questions ask for two things at once: `show me a zebra and something to eat`
- An LLM splits the question into two search requests. Each one searches the images.
- A vision model answers from the retrieved photos and cites the photo IDs
- New columns fill in for questions already in the table

In [ ]:
!cat bonus_answer.py

In [ ]:
!pxt schema diff bonus_answer.py rag_demo

In [ ]:
!pxt schema update bonus_answer.py rag_demo -f

In [ ]:
queries = pxt.get_table('rag_demo.queries')  # reload to see the new columns
queries.insert([{'raw': 'show me a zebra and something to eat'}])
queries.select(queries.raw, queries.parts, queries.photos_1, queries.photos_2, queries.answer).collect()

- ✅ One insert runs the whole chain: rewrite, split, search, answer
- Each answer sits next to the question and the photos it used

## Choose a recipe

- Start with Recipe 1. Stop when your questions get good results.
- Misses because of wording: rewrite the question (Recipe 2)
- Misses because the words aren't in the data: captions and embeddings (Recipe 3)
- Each recipe is a column, an index, or a view on the same tables